# 🚬 Cigarette Detection with YOLOv8
**Author:** Nadeem Gohar

Detects cigarettes in images/video — for flagging smoking in restricted or public areas. Built with Ultralytics YOLOv8 (nano variant for speed + edge-friendliness).

**Pipeline:**
1. Install dependencies
2. Mount Google Drive (so training progress is never lost)
3. Download an annotated cigarette dataset from Roboflow
4. Train YOLOv8
5. Validate (mAP, precision, recall)
6. Run inference on a test image
7. Export `best.pt` for the Streamlit app

> Runtime: Colab → set **Runtime > Change runtime type > GPU (T4)** before running.

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics roboflow

import ultralytics
ultralytics.checks()

## 2. Mount Google Drive (important!)

This saves your training progress to Drive as it goes. If Colab disconnects mid-training, you won't lose anything — you can resume from the last checkpoint instead of starting over.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/cigarette_detection_runs"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Checkpoints will be saved to:", SAVE_DIR)

## 3. Download a cigarette dataset from Roboflow

Go pick a dataset yourself so the workspace/project/version is guaranteed correct:

1. Open one of these in a new tab:
   - https://universe.roboflow.com/yolo-pdvpx/cigarette-h2p1m
   - https://universe.roboflow.com/cigarette-c6554/cigarette-ghnlk (has `face` + `cigarette` classes)
   - Or search yourself: https://universe.roboflow.com/search?q=class%3Acigarette
2. Click **Download Dataset** → format **YOLOv8** → choose **"show download code"** (not raw zip).
3. Copy the code snippet Roboflow shows you and paste it in below, replacing the placeholder.

Paste your **free** Roboflow API key first (get one at https://app.roboflow.com → Settings → API Key).

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "PASTE_YOUR_API_KEY_HERE"  # <-- replace this

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# PASTE the snippet Roboflow gives you here, replacing the two lines below:
project = rf.workspace("yolo-pdvpx").project("cigarette-h2p1m")
dataset = project.version(1).download("yolov8")

print("Dataset location:", dataset.location)

### If your chosen dataset has extra classes (e.g. `face` + `cigarette`), keep only `cigarette`
Run this cell to filter labels down to cigarette-only. Skip it if the dataset already contains only the cigarette class.

In [ ]:
import yaml, os

FILTER_CIGARETTE_ONLY = False  # set True if your dataset has extra classes like 'face'

yaml_path = os.path.join(dataset.location, "data.yaml")
with open(yaml_path) as f:
    data_cfg = yaml.safe_load(f)
print("Classes in dataset:", data_cfg["names"])

if FILTER_CIGARETTE_ONLY and "cigarette" in data_cfg["names"]:
    cig_idx = data_cfg["names"].index("cigarette")
    for split in ["train", "valid", "test"]:
        label_dir = os.path.join(dataset.location, split, "labels")
        if not os.path.isdir(label_dir):
            continue
        for fname in os.listdir(label_dir):
            fpath = os.path.join(label_dir, fname)
            with open(fpath) as f:
                lines = [l for l in f.readlines() if l.split() and l.split()[0] == str(cig_idx)]
            lines = ["0 " + " ".join(l.split()[1:]) + "\n" for l in lines]
            with open(fpath, "w") as f:
                f.writelines(lines)
    data_cfg["names"] = ["cigarette"]
    data_cfg["nc"] = 1
    with open(yaml_path, "w") as f:
        yaml.dump(data_cfg, f)
    print("Filtered to cigarette-only class.")

## 4. Train YOLOv8

In [ ]:
from ultralytics import YOLO

# yolov8n = nano (fastest, smallest, great for edge/deployment)
# yolov8s = small (a bit more accurate, still fast) -- use this if you have GPU time
model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,             # reduced so it's more likely to finish in one Colab session
    imgsz=640,
    batch=16,
    patience=15,           # early stopping if no improvement
    optimizer="AdamW",
    lr0=0.001,
    augment=True,
    project=SAVE_DIR,      # saves to Google Drive, not temporary Colab storage
    name="yolov8n_cigarette",
    exist_ok=True
)

### If Colab disconnects mid-training, run this cell instead of re-running training from scratch

Skip this cell if training above completed normally. Only run this if your session dropped partway through — it picks up exactly where you left off, using the checkpoint already saved in Drive.

In [ ]:
from ultralytics import YOLO

# after re-mounting Drive (cell above), point this to your last checkpoint:
resume_model = YOLO(f"{SAVE_DIR}/yolov8n_cigarette/weights/last.pt")
results = resume_model.train(resume=True)

## 5. Validate the model

In [ ]:
metrics = model.val()
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

## 6. Run inference on a test image

In [ ]:
import glob
from IPython.display import Image, display

test_images = glob.glob(f"{dataset.location}/test/images/*.jpg")[:3]

for img_path in test_images:
    results = model.predict(source=img_path, conf=0.35, save=True)
    for r in results:
        save_path = r.save_dir + "/" + img_path.split("/")[-1]
        display(Image(filename=save_path))

## 7. Export the trained model
Download `best.pt` — this is what powers the Streamlit app.

In [ ]:
from google.colab import files

best_model_path = f"{SAVE_DIR}/yolov8n_cigarette/weights/best.pt"
files.download(best_model_path)
print("Copy this file into your Streamlit app folder as: model/best.pt")
print("It's also safely backed up in your Google Drive at:", best_model_path)

## Notes on accuracy/efficiency tradeoffs
- **yolov8n**: ~3.2M params, fastest inference (great for Streamlit Cloud's limited CPU), slightly lower mAP.
- **yolov8s**: ~11M params, noticeably better mAP, still real-time on GPU, a bit heavier on CPU-only Streamlit Cloud.
- If mAP is low, the usual first fixes are: more epochs, more training images, or heavier augmentation.